In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

In [ ]:
# Load environment variables from the .env file.
# The OpenAI API key is required to generate embeddings.
from dotenv import load_dotenv
load_dotenv(override=True)

In [ ]:
# Verify that the OpenAI API key has been loaded correctly.

import os
print("KEY:", os.getenv("OPENAI_API_KEY"))

In [ ]:
import pandas as pd

books = pd.read_csv("books_cleaned.csv")

In [ ]:
books["tagged_description"]

In [ ]:
# Export the tagged descriptions to a text file.
# Each line represents one document that will later
# be embedded and indexed by Chroma.
books["tagged_description"].to_csv("tagged_description.txt", sep="\n", index=False, header=False)

In [ ]:
# Load the exported text file and split it into
# individual documents for embedding generation.
raw_documents = TextLoader("tagged_description.txt",encoding="utf-8").load()
text_splitter = CharacterTextSplitter(chunk_size=0, chunk_overlap=0, separator="\n")
documents = text_splitter.split_documents(raw_documents)

In [ ]:
# Create OpenAI embeddings for each document and
# store them inside a Chroma vector database.

embedding_model = OpenAIEmbeddings()

db_books = Chroma(embedding_function=embedding_model)

batch_size = 20

for i in range(0, len(documents), batch_size):
    batch = documents[i:i+batch_size]
    db_books.add_documents(batch)

In [ ]:
import pandas as pd
def retrieve_semantic_recommendations(
        query: str,
        top_k: int = 5,
) -> pd.DataFrame:
    """
    Retrieve books whose descriptions are semantically
    similar to the user's query.

    Parameters
    ----------
    query : str
        Natural language search query.

    top_k : int
        Number of most similar documents to retrieve.

    Returns
    -------
    pandas.DataFrame
        DataFrame containing the recommended books.
    """

    recommendations  = db_books.similarity_search(query, top_k)

    books_list = []
    recommended_isbns = [ int(doc.page_content.strip('"').split()[0]) for doc in recommendations ]
    return books[books["isbn13"].isin(recommended_isbns)]

In [ ]:
# Test the semantic retrieval pipeline with a
# natural language query.
retrieve_semantic_recommendations("A book about war")